# EDA

In [1]:
#data
import pandas as pd
#plot
import matplotlib.pyplot as plt
import seaborn as sns
# interactive
from ipywidgets import interact, widgets
# dimentionality reduction
from sklearn.decomposition import PCA
# scalar funtions
from sklearn.preprocessing import StandardScaler
# math
import numpy as np

### Importing data

In [2]:
df_iden = pd.read_csv('data_ieee/train_identity.csv')
df_tran = pd.read_csv('data_ieee/train_transaction.csv')

debug = False 

if debug:
    
    print("--- Columns in File 1 ---")
    for col in df_id.columns:
        print(col)

    print("\n--- Columns in File 2 ---")
    for col in df_tr.columns:
        print(col)

        

## Data filtering

In [3]:

# colum selection
cols_id = [c for c in df_iden.columns if any(x in c[0:3] for x in ['id'])]
cols_dev = ['DeviceType','DeviceInfo']


cols_V = [c for c in df_tran.columns if any(x in c[0:2] for x in ['V'])]
cols_M = [c for c in df_tran.columns if any(x in c[0:2] for x in ['M'])]
cols_D = [c for c in df_tran.columns if any(x in c[0:2] for x in ['D'])]
cols_C = [c for c in df_tran.columns if any(x in c[0:2] for x in ['C'])]
cols_card = [c for c in df_tran.columns if any(x in c for x in ['card'])]

cols_add1 = ['addr1','addr2','dist1','dist2','P_emaildomain','R_emaildomain']
cols_add2 = ['TransactionDT','TransactionAmt','ProductCD']

target = ['isFraud']
join_feature = ['TransactionID']


voted_df_tran = df_tran.groupby(join_feature[0])[target[0]].agg(lambda x: x.mode().iat[0]).reset_index()

# inner join
df_iden_target = pd.merge(df_iden, voted_df_tran, on=join_feature[0], how='inner')




In [4]:
df_iden_target.head()

,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo,isFraud
0,2987004,0.0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M,0
1,2987008,-5.0,98945.0,NaN,NaN,0.0,-5.0,NaN,NaN,NaN,...,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device,0
2,2987010,-5.0,191631.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,NaN,NaN,NaN,F,F,T,T,desktop,Windows,0
3,2987011,-5.0,221832.0,NaN,NaN,0.0,-6.0,NaN,NaN,NaN,...,NaN,NaN,NaN,F,F,T,T,desktop,NaN,0
4,2987016,0.0,7460.0,0.0,0.0,1.0,0.0,NaN,NaN,0.0,...,24.0,1280x800,match_status:2,T,F,T,T,desktop,MacOS,0


### Dataframes distict data

In [5]:
# Dataframes with selected features

df_id = df_iden_target[cols_id + target]
df_dev = df_iden_target[cols_dev + target]

df_V = df_tran[cols_V + target]
df_M = df_tran[cols_M + target]
df_D = df_tran[cols_D + target]
df_C = df_tran[cols_C + target]
df_card = df_tran[cols_card + target]
df_add1 = df_tran[cols_add1 + target]
df_add2 = df_tran[cols_add2 + target]

# Categorical

## Transaction

In [6]:
df_tran.loc[:,[col for col, dtype in df_tran.dtypes.items() if dtype == "str" or dtype == "int"]]

,TransactionID,isFraud,TransactionDT,card1
0,2987000,0,86400,13926
1,2987001,0,86401,2755
2,2987002,0,86469,4663
3,2987003,0,86499,18132
4,2987004,0,86506,4497
...,...,...,...,...
590535,3577535,0,15811047,6550
590536,3577536,0,15811049,10444
590537,3577537,0,15811079,12037
590538,3577538,0,15811088,7826


In [7]:
df_tran['card1'].value_counts()

card1
7919     14932
9500     14162
15885    10361
17188    10344
15066     7945
         ...  
7914         1
15126        1
2570         1
9115         1
1099         1
Name: count, Length: 13553, dtype: int64

In [8]:
df_tran['card1'].unique() # Seems to be a categorical variable

array([13926,  2755,  4663, ..., 13166,  8767, 18038], shape=(13553,))

In [9]:
len(df_tran['TransactionID'].unique()) == len(df_tran)# ID is unique

True

In [10]:
df_tran.TransactionDT # it is the time of the transaction in seconds since the a reference time, not categorical

0            86400
1            86401
2            86469
3            86499
4            86506
            ...   
590535    15811047
590536    15811049
590537    15811079
590538    15811088
590539    15811131
Name: TransactionDT, Length: 590540, dtype: int64

## Identity

In [11]:
df_iden[[col for col, dtype in df_iden.dtypes.items() if dtype == "str" or dtype == "int"]]

,TransactionID
0,2987004
1,2987008
2,2987010
3,2987011
4,2987016
...,...
144228,3577521
144229,3577526
144230,3577529
144231,3577531


## Null Exploration

In [12]:

def nan_count(df):
    nan_counts = df.isna().sum().sort_values(ascending=True)
    
    total_rows = len(df)
    total_columns = len(df.columns)
    

    vertical_size = max(5, int(total_columns / 5)) 

    plt.figure(figsize=(10, vertical_size))
    

    nan_counts.plot(kind='barh', color='skyblue', edgecolor='black')

    plt.axvline(x=total_rows, color='red', linestyle='--', label=f'Total Rows ({total_rows})')

    plt.title('Missing Values per Column (Ordered)', fontsize=14)
    plt.xlabel('Count of NaNs')
    plt.ylabel('Column Name')
    plt.legend()
    plt.grid(axis='x', linestyle=':', alpha=0.7)

    plt.tight_layout()
    plt.show()


In [ ]:
dict_options = {
'df_id'   : df_id,
'df_dev'  : df_dev,
'df_V'    : df_V,
'df_M'    : df_M,
'df_D'    : df_D,
'df_C'    : df_C,
'df_card' : df_card,
'df_add1' : df_add1,
'df_add2' : df_add2
}
dropdown_options = dict_options.keys()

interact(lambda idx: nan_count(dict_options[idx]), idx=widgets.Dropdown(
    options=dropdown_options,
    description='Select data_frame:',
))

interactive(children=(Dropdown(description='Select data_frame:', options=('df_id', 'df_dev', 'df_V', 'df_M', '…

<function __main__.<lambda>(idx)>

## Data Exploration

In [16]:
interact(lambda idx: print(dict_options[idx].describe(include='all')), idx=widgets.Dropdown(
    options=dropdown_options,
    description='Select data_frame:',
))

interactive(children=(Dropdown(description='Select data_frame:', options=('df_id', 'df_dev', 'df_V', 'df_M', '…

<function __main__.<lambda>(idx)>

## V_# feature exploration

In [17]:

nan_V_counts = df_V.isna().sum().sort_values(ascending=True)


### null grouping

In [18]:
nan_series = df_V.drop(columns=target[0],axis=0).isna().sum()
unique_counts = nan_series.unique()
set_V = {}
for count in sorted(unique_counts):
    cols = nan_series[nan_series == count].index.tolist()
    print(f"Count {count}: {cols[:5]}...") 
    set_V[f'Count {count}'] = cols

Count 12: ['V279', 'V280', 'V284', 'V285', 'V286']...
Count 314: ['V95', 'V96', 'V97', 'V98', 'V99']...
Count 1269: ['V281', 'V282', 'V283', 'V288', 'V289']...
Count 76073: ['V12', 'V13', 'V14', 'V15', 'V16']...
Count 77096: ['V53', 'V54', 'V55', 'V56', 'V57']...
Count 89164: ['V75', 'V76', 'V77', 'V78', 'V79']...
Count 168969: ['V35', 'V36', 'V37', 'V38', 'V39']...
Count 279287: ['V1', 'V2', 'V3', 'V4', 'V5']...
Count 449124: ['V220', 'V221', 'V222', 'V227', 'V234']...
Count 450721: ['V169', 'V170', 'V171', 'V174', 'V175']...
Count 450909: ['V167', 'V168', 'V172', 'V173', 'V176']...
Count 460110: ['V217', 'V218', 'V219', 'V223', 'V224']...
Count 508189: ['V322', 'V323', 'V324', 'V325', 'V326']...
Count 508589: ['V143', 'V144', 'V145', 'V150', 'V151']...
Count 508595: ['V138', 'V139', 'V140', 'V141', 'V142']...


### PCA analisis of V_#

In [19]:

results_pca = {}

for group_name, cols in set_V.items():

    df_temp = df_V[cols].copy()
    
    df_temp = df_temp.dropna()
    
    if df_temp.empty:
        print(f"Saltando {group_name}: No quedan filas tras dropna.")
        continue


    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_temp)
    

    pca = PCA(n_components=0.90)
    pca.fit(scaled_data)
    
    results_pca[group_name] = {
        'original_dims': len(cols),
        'pca_dims': pca.n_components_,
        'rows_remaining': len(df_temp)
    }
    
    print(f"{group_name}: De {len(cols)} columnas a {pca.n_components_} para 90% varianza.")



Count 12: De 32 columnas a 8 para 90% varianza.
Count 314: De 43 columnas a 12 para 90% varianza.
Count 1269: De 11 columnas a 6 para 90% varianza.
Count 76073: De 23 columnas a 8 para 90% varianza.
Count 77096: De 22 columnas a 8 para 90% varianza.
Count 89164: De 20 columnas a 8 para 90% varianza.
Count 168969: De 18 columnas a 7 para 90% varianza.
Count 279287: De 11 columnas a 6 para 90% varianza.
Count 449124: De 16 columnas a 5 para 90% varianza.
Count 450721: De 19 columnas a 7 para 90% varianza.
Count 450909: De 31 columnas a 5 para 90% varianza.
Count 460110: De 46 columnas a 8 para 90% varianza.
Count 508189: De 18 columnas a 4 para 90% varianza.
Count 508589: De 11 columnas a 2 para 90% varianza.
Count 508595: De 18 columnas a 5 para 90% varianza.


# C_# feature exploration

In [20]:
def fn_col_group(df):
    return [col for col, dtype in df.dtypes.items() if dtype == 'str' or dtype == 'int']

def filter_col_group(df, col_group):
    col_group_copy = col_group.copy()
    for col in col_group:
        if df[col].nunique() > 10:
            col_group_copy.remove(col)
            print(f'{col} removed {df[col].nunique()} unique values')
    col_group_copy.remove('isFraud')
    return col_group_copy

def fn_catplot(df, col_group):
    if len(col_group) == 0:
        print('No columns to plot')
        return
    fig, axs = plt.subplots(len(col_group), 1, figsize=(5, len(col_group)*3))
    for ax, col in enumerate(col_group):
        if isinstance(axs, np.ndarray):
            sns.countplot(data=df, x=df[col], ax=axs[ax], hue='isFraud')
        else:
            sns.countplot(data=df, x=df[col], ax=axs, hue='isFraud')
    plt.tight_layout()

In [21]:
col_group = fn_col_group(df_C)
col_group = filter_col_group(df_C, col_group)
fn_catplot(df_C, col_group)


No columns to plot


In [22]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_C)

pca = PCA(n_components=0.90)
pca.fit(scaled_data)

f"De {len(df_C.columns)} columnas a {pca.n_components_} para 90% varianza."

'De 15 columnas a 3 para 90% varianza.'

In [23]:
# import prince

# famd = prince.FAMD(
#     n_components=2,
#     n_iter=3,
#     copy=True,
#     check_input=True,
#     random_state=42,
#     engine="sklearn",
#     handle_unknown="error"  # same parameter as sklearn.preprocessing.OneHotEncoder
# )

# df_temp = df_card.dropna().iloc[:len(df_card)//5, :]

# famd = famd.fit(df_temp)

# famd.eigenvalues_summary

# Card_# Feature exploration

In [24]:
col_group = fn_col_group(df_card)
col_group = filter_col_group(df_card, col_group)
fn_catplot(df_card, col_group)

card1 removed 13553 unique values
No columns to plot


# ID_# Feature exploration

In [25]:
df_id[[col for col, dtype in df_id.dtypes.items() if dtype == "str" or dtype == "int"]]

,isFraud
0,0
1,0
2,0
3,0
4,0
...,...
144228,0
144229,1
144230,0
144231,0


In [26]:
col_group = fn_col_group(df_id)
col_group = filter_col_group(df_id, col_group)
fn_catplot(df_id, col_group)

No columns to plot


# Dev Feature exploration

In [27]:
df_dev

,DeviceType,DeviceInfo,isFraud
0,mobile,SAMSUNG SM-G892A Build/NRD90M,0
1,mobile,iOS Device,0
2,desktop,Windows,0
3,desktop,NaN,0
4,desktop,MacOS,0
...,...,...,...
144228,mobile,F3111 Build/33.3.A.1.97,0
144229,mobile,A574BL Build/NMF26F,1
144230,mobile,Moto E (4) Plus Build/NMA26.42-152,0
144231,desktop,MacOS,0


In [28]:
col_group = fn_col_group(df_dev)
col_group = filter_col_group(df_dev, col_group)
fn_catplot(df_dev, col_group)

No columns to plot


# M_# Feature exploration

In [29]:
df_M

,M1,M2,M3,M4,M5,M6,M7,M8,M9,isFraud
0,T,T,T,M2,F,T,NaN,NaN,NaN,0
1,NaN,NaN,NaN,M0,T,T,NaN,NaN,NaN,0
2,T,T,T,M0,F,F,F,F,F,0
3,NaN,NaN,NaN,M0,T,F,NaN,NaN,NaN,0
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...
590535,T,T,T,M0,T,F,F,F,T,0
590536,T,F,F,M0,F,T,F,F,F,0
590537,T,F,F,NaN,NaN,T,NaN,NaN,NaN,0
590538,T,T,T,M0,F,T,NaN,NaN,NaN,0


In [30]:
col_group = fn_col_group(df_M)
col_group = filter_col_group(df_M, col_group)
fn_catplot(df_M, col_group)

No columns to plot


# D_# Feature exploration

In [31]:
df_D

,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10,D11,D12,D13,D14,D15,isFraud
0,14.0,NaN,13.0,NaN,NaN,NaN,NaN,NaN,NaN,13.0,13.0,NaN,NaN,NaN,0.0,0
1,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,0
2,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,315.0,NaN,NaN,NaN,315.0,0
3,112.0,112.0,0.0,94.0,0.0,NaN,NaN,NaN,NaN,84.0,NaN,NaN,NaN,NaN,111.0,0
4,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,29.0,29.0,30.0,NaN,NaN,NaN,NaN,NaN,NaN,56.0,56.0,NaN,NaN,NaN,56.0,0
590536,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0
590537,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,0
590538,22.0,22.0,0.0,22.0,0.0,NaN,NaN,NaN,NaN,22.0,22.0,NaN,NaN,NaN,22.0,0


In [32]:
col_group = fn_col_group(df_D)
col_group = filter_col_group(df_D, col_group)
fn_catplot(df_D, col_group)

No columns to plot


# Addr1 Feature exploration

In [33]:
df_add1

,addr1,addr2,dist1,dist2,P_emaildomain,R_emaildomain,isFraud
0,315.0,87.0,19.0,NaN,NaN,NaN,0
1,325.0,87.0,NaN,NaN,gmail.com,NaN,0
2,330.0,87.0,287.0,NaN,outlook.com,NaN,0
3,476.0,87.0,NaN,NaN,yahoo.com,NaN,0
4,420.0,87.0,NaN,NaN,gmail.com,NaN,0
...,...,...,...,...,...,...,...
590535,272.0,87.0,48.0,NaN,NaN,NaN,0
590536,204.0,87.0,NaN,NaN,gmail.com,NaN,0
590537,231.0,87.0,NaN,NaN,gmail.com,NaN,0
590538,387.0,87.0,3.0,NaN,aol.com,NaN,0


In [34]:
col_group = fn_col_group(df_add1)
col_group = filter_col_group(df_add1, col_group)
fn_catplot(df_add1, col_group)

No columns to plot


# Add2 Feature exploration

In [35]:
df_add2

,TransactionDT,TransactionAmt,ProductCD,isFraud
0,86400,68.50,W,0
1,86401,29.00,W,0
2,86469,59.00,W,0
3,86499,50.00,W,0
4,86506,50.00,H,0
...,...,...,...,...
590535,15811047,49.00,W,0
590536,15811049,39.50,W,0
590537,15811079,30.95,W,0
590538,15811088,117.00,W,0


In [36]:
col_group = fn_col_group(df_add2)
col_group = filter_col_group(df_add2, col_group)
fn_catplot(df_add2, col_group)

TransactionDT removed 573349 unique values
No columns to plot
